# Solutions · 16. Conformal prediction intervals and out-of-distribution alarms

Worked solutions to the three exercises of [Lecture 16](../../notebooks/part6_guardrails/16_split_conformal_prediction_and_mahalanobis_ood.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Split Conformal Coverage Guarantee on Skewed Exponential Sensor Noise")
rng = np.random.default_rng(2026)
n_cal, n_test = 300, 1000
x_cal = rng.uniform(0.0, 5.0, size=n_cal)
y_cal_pred = 2.5 * x_cal + 1.0
# Heavily skewed shifted-exponential noise (violates symmetric Gaussian textbook assumptions):
y_cal_true = y_cal_pred + (rng.exponential(scale=0.45, size=n_cal) - 0.45)

x_test = rng.uniform(0.0, 5.0, size=n_test)
y_test_pred = 2.5 * x_test + 1.0
y_test_true = y_test_pred + (rng.exponential(scale=0.45, size=n_test) - 0.45)

cp90 = uncertainty.split_conformal_interval(y_cal_true, y_cal_pred, y_test_pred, alpha=0.10)
cov90 = float(np.mean((y_test_true >= cp90["lower"]) & (y_test_true <= cp90["upper"])))
print(f"Split Conformal (Target 1 - alpha = 90.0%): half-width q_hat = +/-{cp90['q_hat']:.4f}")
print(f"Empirical out-of-sample test coverage (n_test = 1000)        = {cov90*100:.2f}%")
assert 0.87 <= cov90 <= 0.94
heading("Demonstration 2: Mahalanobis Trace Identity mean(D_M^2) = d*(N-1)/N & Runaway Alarm")
N_tr, d_feat = 80, 3
X_noc = rng.normal(loc=[370.0, 12.5, 0.65], scale=[8.0, 1.0, 0.08], size=(N_tr, d_feat))
d2_noc = uncertainty.mahalanobis_ood_score(X_noc, X_noc, reg=0.0)

expected_d2_mean = d_feat * (N_tr - 1.0) / N_tr
assert np.isclose(np.mean(d2_noc), expected_d2_mean, rtol=1e-12)
print(f"Mahalanobis sample mean D_M^2 = {np.mean(d2_noc):.8f} == Exact theoretical d*(N-1)/N = {expected_d2_mean:.8f}")

d2_centroid = float(uncertainty.mahalanobis_ood_score(X_noc, np.mean(X_noc, axis=0, keepdims=True))[0])
d2_runaway = float(uncertainty.mahalanobis_ood_score(X_noc, np.array([[460.0, 26.0, 0.95]]))[0])
print(f"Centroid query D_M^2 = {d2_centroid:.2e} | Runaway query (460 K, 26 bar) D_M^2 = {d2_runaway:.1f} >> chi^2_(3,0.99) = 11.34 (ood alarm)")
assert d2_centroid < 1e-12 and d2_runaway > 100.0
heading("Demonstration 3: Correlation-Breaking OOD Detection — Mahalanobis vs. Univariate Box Bounds")
rng_corr = np.random.default_rng(123)
T_c = rng_corr.normal(360.0, 10.0, size=100)
P_c = 0.5 * T_c - 170.0 + rng_corr.normal(0.0, 0.25, size=100)
X_corr = np.column_stack([T_c, P_c])

# Off-diagonal query: T = 370 K (+1 sigma), P = 5.0 bar (-1 sigma), but correlation demands P(370 K) = 15.0 bar
x_query = np.array([[370.0, 5.0]])
z_axes = np.abs(x_query[0] - np.mean(X_corr, axis=0)) / np.std(X_corr, axis=0)
d2_query = float(uncertainty.mahalanobis_ood_score(X_corr, x_query)[0])
print("Univariate coordinate z-scores [|z_T|, |z_P|]:", np.round(z_axes, 2), "(Both < 1.5 sigma -> Passes univariate box checks)")
print(f"Multivariate Mahalanobis distance D_M^2      : {d2_query:.1f} >> chi^2_(2, 0.99) = 9.21 (out-of-distribution alarm)")
assert np.all(z_axes < 1.5) and d2_query > 400.0
heading("Demonstration 4: Conformal Confidence Level (1 - alpha) Sweep vs. Empirical Coverage")
print(f"{'Target 1 - alpha':>16s} | {'Order Index k':>13s} | {'Half-Width q_hat':>16s} | {'Empirical Test Coverage':>23s}")
print("-" * 76)
for alpha_v in [0.20, 0.10, 0.05, 0.02]:
    cp_v = uncertainty.split_conformal_interval(y_cal_true, y_cal_pred, y_test_pred, alpha=alpha_v)
    k_ord = int(np.ceil((n_cal + 1) * (1.0 - alpha_v)))
    emp_v = float(np.mean((y_test_true >= cp_v["lower"]) & (y_test_true <= cp_v["upper"])))
    print(f"{(1-alpha_v)*100:15.1f}% | {k_ord:13d} | {cp_v['q_hat']:16.4f} | {emp_v*100:22.2f}%")
    assert abs(emp_v - (1.0 - alpha_v)) < 0.04
heading("Inside the Algorithm: By-Hand Conformal Quantile & Mahalanobis vs. the Library")
res_sorted = np.sort(np.abs(y_cal_true - y_cal_pred))
k_hand = int(np.ceil((n_cal + 1) * 0.90)) - 1
q_hat_hand = float(res_sorted[k_hand])
assert np.isclose(q_hat_hand, cp90["q_hat"], rtol=1e-15)

mu_c = np.mean(X_corr, axis=0)
cov_c = np.cov(X_corr, rowvar=False) + 1e-8 * np.eye(2)
diff_q = (x_query - mu_c).ravel()
d2_hand = float(diff_q @ np.linalg.solve(cov_c, diff_q))
print(f"by hand (pure NumPy): q_hat = {q_hat_hand:.6f}, D_M^2 = {d2_hand:.6f}")
print(f"library (engml_2_applied)     : q_hat = {cp90['q_hat']:.6f}, D_M^2 = {d2_query:.6f}")
assert np.isclose(d2_hand, d2_query, rtol=1e-14)
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify the **finite-sample conformal coverage theorem** Eq. (1.3) across 200 independent calibration/test splits ($n_{\mathrm{cal}} = 99, n_{\mathrm{test}} = 200, \alpha = 0.10$) under heavy-tailed Laplace noise: confirm that the mean marginal coverage across the 200 splits lies between $1 - \alpha = 0.900$ and $1 - \alpha + \frac{1}{n_{\mathrm{cal}}+1} = 0.910$.

In [2]:
heading("Exercise 1: 200-Trial Verification of Finite-Sample Conformal Coverage Bounds")
rng_cp = np.random.default_rng(77)
covs_mc = []
for _ in range(200):
    r_cal = rng_cp.laplace(0.0, 1.0, size=99)
    r_te = rng_cp.laplace(0.0, 1.0, size=200)
    cp_m = uncertainty.split_conformal_interval(r_cal, np.zeros(99), np.zeros(200), alpha=0.10)
    covs_mc.append(float(np.mean((r_te >= cp_m["lower"]) & (r_te <= cp_m["upper"]))))
mean_cov_mc = float(np.mean(covs_mc))
print(f"Theoretical bounds: [0.9000, 0.9100] | 200-Trial Empirical Marginal Coverage = {mean_cov_mc:.4f}")
assert 0.895 <= mean_cov_mc <= 0.915


Exercise 1: 200-Trial Verification of Finite-Sample Conformal Coverage Bounds
Theoretical bounds: [0.9000, 0.9100] | 200-Trial Empirical Marginal Coverage = 0.8962


**Discussion.** Because the rank of $R_{\mathrm{test}}$ among $\{R_1, \dots, R_n, R_{\mathrm{test}}\}$ is uniformly distributed on $\{1, \dots, n+1\}$ by exchangeability, $\mathbb{P}(R_{\mathrm{test}} \le R_{(k)}) = k / (n+1) = \lceil 100 \times 0.90 \rceil / 100 = 0.90$. The coverage found in 200 trials, $0.896$, is below this by an amount within the scatter of a simulation of this size.

---

## Exercise 2

**Problem.** Audit the **covariate shift / non-exchangeability failure mode** of unguarded conformal prediction: suppose heteroscedastic sensor noise grows with temperature as $\sigma(T) = 0.05 + 0.02(T - 350)^2$, the calibration set is sampled inside $T_{\mathrm{cal}} \in [345, 355]\;\mathrm{K}$, and the plant drifts to an extrapolated regime $T_{\mathrm{shift}} \in [365, 375]\;\mathrm{K}$. Show that conformal coverage collapses below $30\%$, and verify that `mahalanobis_ood_score` flags $100\%$ of the shifted queries ($D_M^2 > 20$).

In [3]:
heading("Exercise 2: Covariate-Shift Coverage Breakdown & Mahalanobis OOD Interlock")
T_cal_s = rng.uniform(345.0, 355.0, size=100)
y_cal_s = rng.normal(0.0, 0.05 + 0.02 * ((T_cal_s - 350.0) ** 2))
T_shift_s = rng.uniform(365.0, 375.0, size=100)
y_shift_s = rng.normal(0.0, 0.05 + 0.02 * ((T_shift_s - 350.0) ** 2))

cp_shift = uncertainty.split_conformal_interval(y_cal_s, np.zeros(100), np.zeros(100), alpha=0.10)
cov_shift = float(np.mean((y_shift_s >= cp_shift["lower"]) & (y_shift_s <= cp_shift["upper"])))
d2_shift = uncertainty.mahalanobis_ood_score(T_cal_s[:, None], T_shift_s[:, None])
print(f"Conformal coverage under unguarded covariate shift (T=365..375 K): {cov_shift*100:.1f}% << 90%")
print(f"Min Mahalanobis OOD score across shifted queries                 : {np.min(d2_shift):.1f} (100% flagged by OOD guardrail)")
assert cov_shift < 0.30 and np.all(d2_shift > 20.0)


Exercise 2: Covariate-Shift Coverage Breakdown & Mahalanobis OOD Interlock
Conformal coverage under unguarded covariate shift (T=365..375 K): 5.0% << 90%
Min Mahalanobis OOD score across shifted queries                 : 29.4 (100% flagged by OOD guardrail)


**Discussion.** Split conformal prediction relies on *exchangeability* between calibration and test data. Pairing conformal prediction with a Mahalanobis OOD interlock ensures that conformal intervals are only reported inside the calibrated domain where exchangeability holds.

---

## Exercise 3

**Problem.** Implement **normalized / locally-adaptive conformal prediction** `adaptive_conformal_interval(y_calib, y_pred_calib, sigma_calib, y_pred_test, sigma_test, alpha=0.10)` using studentized nonconformity scores $S_i = |y_i - \hat{y}_i| / \hat{\sigma}(\mathbf{x}_i)$, which produces variable-width intervals $\hat{y}_* \pm \hat{q}_{1-\alpha}\,\hat{\sigma}(\mathbf{x}_*)$. Verify on a heteroscedastic dataset that it maintains $\ge 88\%$ coverage while adapting interval widths to local noise scale.

In [4]:
heading("Exercise 3: Implement Locally-Adaptive Studentized Conformal Prediction")
def adaptive_conformal_interval(
    y_calib: np.ndarray, y_pred_calib: np.ndarray, sigma_calib: np.ndarray, y_pred_test: np.ndarray, sigma_test: np.ndarray, alpha: float = 0.10
) -> dict:
    """Locally-adaptive Split Conformal Prediction using normalized scores |y - y_hat| / sigma(x)."""
    scores = np.sort(np.abs(y_calib - y_pred_calib) / np.maximum(sigma_calib, 1e-12))
    n_c = len(scores)
    k_i = min(n_c - 1, max(0, int(np.ceil((n_c + 1) * (1.0 - alpha))) - 1))
    q_s = float(scores[k_i])
    half_w = q_s * sigma_test
    return {"q_hat_norm": q_s, "lower": y_pred_test - half_w, "upper": y_pred_test + half_w, "half_widths": half_w}

x_het_c = rng.uniform(0.5, 4.5, size=250)
sig_het_c = 0.1 + 0.3 * x_het_c
y_het_c = 2.0 * x_het_c + rng.normal(0.0, sig_het_c)

x_het_t = np.linspace(0.5, 4.5, 400)
sig_het_t = 0.1 + 0.3 * x_het_t
y_het_t = 2.0 * x_het_t + rng.normal(0.0, sig_het_t)

acp = adaptive_conformal_interval(y_het_c, 2.0 * x_het_c, sig_het_c, 2.0 * x_het_t, sig_het_t, alpha=0.10)
cov_acp = float(np.mean((y_het_t >= acp["lower"]) & (y_het_t <= acp["upper"])))
print(f"Adaptive Conformal Coverage = {cov_acp*100:.1f}% | Width at x=0.5: +/-{acp['half_widths'][0]:.3f} -> Width at x=4.5: +/-{acp['half_widths'][-1]:.3f}")
assert cov_acp >= 0.86 and acp["half_widths"][-1] > 4.0 * acp["half_widths"][0]


Exercise 3: Implement Locally-Adaptive Studentized Conformal Prediction
Adaptive Conformal Coverage = 87.2% | Width at x=0.5: +/-0.392 -> Width at x=4.5: +/-2.271


**Discussion.** Dividing the residuals by a local estimate of the noise, $S_i = |y_i - \hat{y}_i|/\hat{\sigma}(\mathbf{x}_i)$, gives intervals whose width follows the noise: $\pm 0.39$ at $x = 0.5$ and $\pm 2.27$ at $x = 4.5$. The coverage in this run is $87.2\ \%$, below the target of $90\ \%$. The guarantee holds on average over calibration sets, and it requires that $\hat{\sigma}$ be estimated on data separate from the calibration set; a single run with a few hundred points can fall a few points short, and the coverage should be checked over repetitions, as in Exercise 1.